In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata
import os
import seaborn as sb
import matplotlib
from harmony import harmonize

In [2]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

# Ignore R warning messages
#Note: this can be commented out to get more verbose R output
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

# Automatically convert rpy2 outputs to pandas dataframes
pandas2ri.activate()
anndata2ri.activate()
%load_ext rpy2.ipython

/tmp/ipykernel_1293751/2253401465.py:13: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


## ECs

In [3]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_human/Endothelial_cell/level2_ec_pre_annot.rds")
level1_marker <- list(
    "Arterial endothelial cell" = c('GJ4', 'GJ5', "GATA2", "MECOM", "GJA4", "GJA5"),
    "Lymphatic endothelial cell" = c("LYVE1", "CCL21")
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_human/Endothelial_cell/level2_ec_pre_annot.csv",row.names = FALSE, )


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    List of 2
 $ Arterial endothelial cell : chr [1:6] "GJ4" "GJ5" "GATA2" "MECOM" ...
 $ Lymphatic endothelial cell: chr [1:2] "LYVE1" "CCL21"


Loading required package: SeuratObject
Loading required package: sp

Attaching package: ‘SeuratObject’

The following objects are masked from ‘package:base’:

    intersect, t


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



## Macrophage

In [4]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_human/Macrophage/level2_macros_pre_annot.rds")
level1_marker <- list(
    "Inflammatory macrophage" = c('S100A8', 'IL1B', 'S100A9','IRF7', 'IFITM3', 'ISG15', 'IFIT2'),
    "Foamy macrophage" = c('TREM2', 'MARCO', 'FABP4','FABP5', 'CD36'),
    "Homeostatic/Resident macrophage" = c('LYVE1', 'MRC1', 'F13A1', 'SEPP1', 'IGF1', 'GAS6', 'MERTK', 'STAB1', 'C1QA', 'C1QB', 'C1QC', 'FOLR2'),
    "Proliferative macrophage" = c('MKI67', 'TOP2A', 'PCNA', 'BIRC5', 'MCM2','MCM4','MCM5', 'TK1'),
    "Matrix-remodeling/SMC-like macrophage" = c('AEBP1', 'COL1A1', 'COL1A2', 'THBS2', 'MMP3', 'S100A16', 'TNC', 'PDGFRB')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_human/Macrophage/level2_macros_pre_annot.csv",row.names = FALSE, )

List of 5
 $ Inflammatory macrophage              : chr [1:7] "S100A8" "IL1B" "S100A9" "IRF7" ...
 $ Foamy macrophage                     : chr [1:5] "TREM2" "MARCO" "FABP4" "FABP5" ...
 $ Homeostatic/Resident macrophage      : chr [1:12] "LYVE1" "MRC1" "F13A1" "SEPP1" ...
 $ Proliferative macrophage             : chr [1:8] "MKI67" "TOP2A" "PCNA" "BIRC5" ...
 $ Matrix-remodeling/SMC-like macrophage: chr [1:8] "AEBP1" "COL1A1" "COL1A2" "THBS2" ...


## DCs

In [5]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_human/Dendritic_cell/level2_dc_pre_annot.rds")
level1_marker <- list(
    "cDC1" = c("CLEC9A", "IRF8", "SNX3"),
    "cDC2" = c("CD1C", "CLEC10A", "FCER1A"),
    "Plasmacytoid dendritic cell" = c("IRF7", "TLR7", "TLR9", "NRP1", "CLEC4C", "SCAMP5", "GZMB", "TCF4")
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_human/Dendritic_cell/level2_dc_pre_annot.csv",row.names = FALSE, )

List of 3
 $ cDC1                       : chr [1:3] "CLEC9A" "IRF8" "SNX3"
 $ cDC2                       : chr [1:3] "CD1C" "CLEC10A" "FCER1A"
 $ Plasmacytoid dendritic cell: chr [1:8] "IRF7" "TLR7" "TLR9" "NRP1" ...


## T cells

In [6]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_human/T_cell/level2_tc_pre_annot.rds")
level1_marker <- list(
    "CD8 T cell" = c('CCL4L2', 'CRTAM', 'GZMK', 'CCL4', 'CD8A', 'GZMH'),
    "CD4 T cell" = c('IL7R', 'ANXA1', 'FTH1', 'CD4', 'BATF', 'TNFRSF4', 'TNFRSF18')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_human/T_cell/level2_tc_pre_annot.csv",row.names = FALSE, )

List of 2
 $ CD8 T cell: chr [1:6] "CCL4L2" "CRTAM" "GZMK" "CCL4" ...
 $ CD4 T cell: chr [1:7] "IL7R" "ANXA1" "FTH1" "CD4" ...


## B cells

In [10]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_human/B_cell/level2_bc_pre_annot.rds")
level1_marker <- list(
    "B cell" = c('CD79A', 'CD79B', 'MS4A1', 'CD22', 'FCER2'),
    "Plasma cell" = c('IGKC', 'IGHM', 'IGHA1', 'IGLC2', 'IGLC3', 'JCHAIN'),
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_human/B_cell/level2_bc_pre_annot.csv",row.names = FALSE, )

Error in list(`B cell` = c("CD79A", "CD79B", "MS4A1", "CD22", "FCER2"),  : 
  argument 3 is empty


RInterpreterError: Failed to parse and evaluate line 'library(Seurat)\nlibrary(openxlsx)\nlibrary(Matrix)\nlibrary(stringr)\nlibrary(dplyr)\nadata <- readRDS("./output_human/B_cell/level2_bc_pre_annot.rds")\nlevel1_marker <- list(\n    "B cell" = c(\'CD79A\', \'CD79B\', \'MS4A1\', \'CD22\', \'FCER2\'),\n    "Plasma cell" = c(\'IGKC\', \'IGHM\', \'IGHA1\', \'IGLC2\', \'IGLC3\', \'JCHAIN\'),\n)\nstr(level1_marker)\n\nsc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")\nmodule_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)\n#按聚类分组，计算每个模块的平均得分\ncluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%\n  rename(Cluster = leiden)  \ncluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])\ncluster_top_celltype <- gsub("ModuleScore_", "",

## Monocyte

In [8]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_human/Monocyte/level2_mono_pre_annot.rds")
level1_marker <- list(
    "Classical monocyte" = c('CD14', 'FCN1', 'LST1', 'VCAN', 'CCR2', 'SELL', 'CTSS'),
    "Intermediate monocyte" = c('MS4A7', 'HLA-DPA1', 'HLA-DPB1', 'ITGAX', 'IRF7', 'STAT1'),
    "Non-classical monocyte" = c('FCGR3A', 'CX3CR1', 'NR4A1', 'ITGAL', 'PDE4B', 'LILRB1', 'HSPA1A')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_human/Monocyte/level2_mono_pre_annot.csv",row.names = FALSE, )

List of 3
 $ Classical monocyte    : chr [1:7] "CD14" "FCN1" "LST1" "VCAN" ...
 $ Intermediate monocyte : chr [1:6] "MS4A7" "HLA-DPA1" "HLA-DPB1" "ITGAX" ...
 $ Non-classical monocyte: chr [1:7] "FCGR3A" "CX3CR1" "NR4A1" "ITGAL" ...


## SMC

In [9]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_human/SMC/level2_smc_pre_annot.rds")
level1_marker <- list(
    "Smooth muscle cell" = c('MYH11','MYL9','TPM2','CALD1','TAGLN','APOE','APOC1','AGT','NOTCH3','PDGFRB','MFAP4'),
    "Fibromyocyte" = c('FN1','LUM','TNFRSF11B','ACTA2','TCF21')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_human/SMC/level2_smc_pre_annot.csv",row.names = FALSE, )

List of 2
 $ Smooth muscle cell: chr [1:11] "MYH11" "MYL9" "TPM2" "CALD1" ...
 $ Fibromyocyte      : chr [1:5] "FN1" "LUM" "TNFRSF11B" "ACTA2" ...
